In [1]:
#!/usr/bin/env python3
"""
Within-corpus near-duplicate audit  —  CPU only, no GPU, no retraining.

This closes the one remaining reviewer objection that does not need a training run.
Both external reviews flagged that BanglaSarc lost 9.3% of its rows to *exact*
duplication before splitting, an order of magnitude above the other two corpora,
and that its near-duplicate structure *within* train/val/test was never audited.
Its 0.980 in-domain score is the denominator of every retention figure in the paper.

What this does
--------------
1. Builds near-duplicate clusters inside each corpus with character n-gram TF-IDF
   cosine similarity and union-find, at the same 0.92 threshold already used for
   the between-corpus analysis in Section 6.8.
2. Reports cluster statistics per corpus. No one has published these for these
   three corpora, so the table is a contribution in itself.
3. Reports how many test items have a near-duplicate in their own train or
   validation split — the quantity that would inflate the in-domain reference.
4. Writes a leakage-free test subset per corpus so the diagonal can be re-scored
   from saved predictions, no retraining required.

Usage
-----
    python 21_within_corpus_nearduplicate_audit.py

Set NB21_ROOT if the repository is not found automatically.

Runtime: a few minutes on a laptop. Memory scales with the largest corpus
(Ben-Sarc, ~25k rows); the blocked cosine pass keeps peak memory modest.
"""
import os, re, sys, json, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, recall_score

THRESHOLD = 0.92          # same as the between-corpus analysis in Section 6.8
BLOCK = 2000              # rows per similarity block


# ---------------------------------------------------------------- repo
def find_root():
    env = os.getenv("NB21_ROOT", "").strip()
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().resolve().parents]
    for extra in ("/workspace/Sarcasm_detection", "/workspace", "/content"):
        cands.append(Path(extra))
    for c in cands:
        if (c / "01_data" / "interim" / "splits").is_dir():
            return c.resolve()
    raise SystemExit("Could not find the repo. Set NB21_ROOT to the folder "
                     "containing 01_data/interim/splits/")


ROOT = find_root()
SPLITS = ROOT / "01_data" / "interim" / "splits"
OUT = ROOT / "04_outputs" / "finalized_outputs" / "tables"
PRED = ROOT / "04_outputs" / "predictions"
OUT.mkdir(parents=True, exist_ok=True)

CORPORA = ["ben_sarc_binary", "banglasarc_binary", "banglasarc3_binary"]
DISPLAY = {"ben_sarc_binary": "Ben-Sarc", "banglasarc_binary": "BanglaSarc",
           "banglasarc3_binary": "BanglaSarc3"}
_ZW = {ord(c): None for c in ["\u200b", "\u200c", "\u200d", "\ufeff"]}


def norm_key(v):
    if not isinstance(v, str):
        v = "" if pd.isna(v) else str(v)
    v = unicodedata.normalize("NFC", v).translate(_ZW)
    return re.sub(r"\s+", " ", v).strip().casefold()


def strict_key(v):
    """Stricter than the exact key: also collapses punctuation, digits and
    character elongation, matching Section 6.8's stricter filter."""
    v = norm_key(v)
    v = re.sub(r"(.)\1{2,}", r"\1\1", v)      # elongation
    v = re.sub(r"[\d]+", "", v)                # digits
    v = re.sub(r"[^\w\s\u0980-\u09FF]", "", v)  # punctuation, keep Bengali block
    return re.sub(r"\s+", " ", v).strip()


def load(corpus):
    frames = []
    for split in ("train", "val", "test"):
        p = SPLITS / f"{corpus}_{split}.csv"
        if not p.exists():
            raise SystemExit(f"missing {p}")
        d = pd.read_csv(p)
        lab = next(c for c in ["label_binary", "label", "y", "target"] if c in d.columns)
        txt = next(c for c in ["text", "comment", "sentence", "content"] if c in d.columns)
        d = d.rename(columns={lab: "label_binary", txt: "text"})
        d["split"] = split
        d["row_pos"] = np.arange(len(d))
        frames.append(d[["text", "label_binary", "split", "row_pos"]])
    df = pd.concat(frames, ignore_index=True)
    df["norm"] = df.text.map(norm_key)
    df["strict"] = df.text.map(strict_key)
    return df


# ---------------------------------------------------------------- union-find
class UF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def cluster(df, threshold=THRESHOLD):
    """Connected components over pairs with char n-gram cosine >= threshold."""
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                          min_df=2, max_features=200_000, sublinear_tf=True)
    X = vec.fit_transform(df.text.astype(str))
    # L2-normalised by default, so dot product is cosine
    n = X.shape[0]
    uf = UF(n)
    pairs = 0
    for start in range(0, n, BLOCK):
        stop = min(n, start + BLOCK)
        sims = (X[start:stop] @ X.T).toarray()
        for local_i, gi in enumerate(range(start, stop)):
            row = sims[local_i]
            row[gi] = 0.0
            hits = np.flatnonzero(row >= threshold)
            for gj in hits:
                if gj > gi:
                    uf.union(gi, int(gj))
                    pairs += 1
        del sims
    labels = np.array([uf.find(i) for i in range(n)])
    # relabel compactly
    _, comp = np.unique(labels, return_inverse=True)
    return comp, pairs


# ---------------------------------------------------------------- main
print(f"root: {ROOT}\nthreshold: {THRESHOLD}\n")
cluster_rows, leak_rows = [], []

for corpus in CORPORA:
    df = load(corpus)
    print(f"{DISPLAY[corpus]}: {len(df)} rows  "
          f"(train {(df.split=='train').sum()}, val {(df.split=='val').sum()}, "
          f"test {(df.split=='test').sum()})")

    comp, pairs = cluster(df)
    df["cluster"] = comp
    sizes = df.groupby("cluster").size()
    multi = sizes[sizes > 1]

    # how many test items share a cluster with a train or val item?
    tr_val_clusters = set(df.loc[df.split.isin(["train", "val"]), "cluster"])
    test = df[df.split == "test"]
    leaked_mask = test.cluster.isin(tr_val_clusters)
    n_leak = int(leaked_mask.sum())

    # exact-key version of the same question, for comparison
    tv_keys = set(df.loc[df.split.isin(["train", "val"]), "norm"])
    n_leak_exact = int(test.norm.isin(tv_keys).sum())
    tv_strict = set(df.loc[df.split.isin(["train", "val"]), "strict"])
    n_leak_strict = int(test.strict.isin(tv_strict).sum())

    cluster_rows.append(dict(
        corpus=corpus, n_rows=len(df),
        n_clusters=int(sizes.shape[0]),
        n_multi_item_clusters=int(multi.shape[0]),
        largest_cluster=int(sizes.max()),
        rows_in_multi_item_clusters=int(multi.sum()),
        pct_rows_in_clusters=round(100 * multi.sum() / len(df), 2),
        near_dup_pairs=pairs))

    leak_rows.append(dict(
        corpus=corpus, n_test=len(test),
        test_with_exact_dup_in_train_or_val=n_leak_exact,
        test_with_strict_key_dup=n_leak_strict,
        test_in_cluster_with_train_or_val=n_leak,
        pct_test_leaked=round(100 * n_leak / len(test), 2),
        n_test_clean=len(test) - n_leak))

    # write the leakage-free test subset so the diagonal can be re-scored
    clean = test[~leaked_mask]
    keep = pd.DataFrame({"row_pos": clean.row_pos.values,
                         "label_binary": clean.label_binary.values})
    keep.to_csv(OUT / f"21_clean_test_{corpus}.csv", index=False)
    print(f"  clusters {sizes.shape[0]}  multi-item {multi.shape[0]}  "
          f"largest {sizes.max()}  test leaked {n_leak}/{len(test)} "
          f"({100*n_leak/len(test):.1f}%)\n")

cl = pd.DataFrame(cluster_rows)
lk = pd.DataFrame(leak_rows)
cl.to_csv(OUT / "21_within_corpus_clusters.csv", index=False)
lk.to_csv(OUT / "21_within_corpus_leakage.csv", index=False)

print("=" * 78)
print("CLUSTER STRUCTURE")
print(cl.to_string(index=False))
print()
print("TEST-SET LEAKAGE AGAINST OWN TRAIN/VAL")
print(lk.to_string(index=False))
print("=" * 78)

# ---------------------------------------------------------------- re-score
print("\nRe-scoring the in-domain diagonal on the leakage-free test subsets.")
print("This uses saved predictions only; no model is retrained.\n")

pred_dirs = [PRED / "18_multiseed_cross_corpus",
             ROOT / "archives" / "notebook18_sanitized_review_package" / "predictions"]
FN = re.compile(r"^(?:\d+[a-z]?_)?(?P<system>vanilla|fgm)_(?P<source>.+?)_"
                r"seed(?P<seed>\d+)_to_(?P<target>.+?)\.csv$", re.I)

rows = []
for corpus in CORPORA:
    clean = pd.read_csv(OUT / f"21_clean_test_{corpus}.csv")
    keep_pos = set(clean.row_pos.tolist())
    for d in pred_dirs:
        if not d.is_dir():
            continue
        for f in sorted(d.glob("*.csv")):
            m = FN.match(f.name)
            if not m:
                continue
            g = m.groupdict()
            if g["source"] != corpus or g["target"] != corpus:
                continue     # diagonal only
            p = pd.read_csv(f)
            gold = next((c for c in ["gold_label", "label_binary", "label"] if c in p.columns), None)
            pred = next((c for c in ["pred_label", "prediction", "pred"] if c in p.columns), None)
            if gold is None or pred is None:
                continue
            if len(p) != clean.row_pos.max() + 1 and "row_pos" not in p.columns:
                # predictions are in split order for the unfiltered diagonal test set
                p = p.reset_index().rename(columns={"index": "row_pos"})
            elif "row_pos" not in p.columns:
                p = p.reset_index().rename(columns={"index": "row_pos"})
            sub = p[p.row_pos.isin(keep_pos)]
            if len(sub) == 0:
                continue
            rows.append(dict(
                corpus=corpus, system=g["system"], seed=int(g["seed"]),
                n_full=len(p), n_clean=len(sub),
                macro_f1_full=f1_score(p[gold], p[pred], average="macro", zero_division=0),
                macro_f1_clean=f1_score(sub[gold], sub[pred], average="macro", zero_division=0),
                acc_clean=accuracy_score(sub[gold], sub[pred])))
        break

if rows:
    r = pd.DataFrame(rows)
    summ = (r.groupby(["corpus", "system"])
            .agg(seeds=("seed", "nunique"),
                 macro_f1_full=("macro_f1_full", "mean"),
                 macro_f1_clean=("macro_f1_clean", "mean"),
                 n_full=("n_full", "max"), n_clean=("n_clean", "max"))
            .reset_index())
    summ["delta"] = summ.macro_f1_clean - summ.macro_f1_full
    summ.to_csv(OUT / "21_indomain_leakage_free_rescore.csv", index=False)
    print(summ.to_string(index=False))
    worst = summ.loc[summ.delta.idxmin()]
    print(f"\nLargest drop: {DISPLAY.get(worst.corpus, worst.corpus)} "
          f"({worst.system}) {worst.delta:+.4f}")
    print("\nHow to read this. Removing test items that share a near-duplicate cluster")
    print("with their own train or validation split bounds how much the in-domain")
    print("reference is inflated by within-corpus redundancy. It is weaker than a full")
    print("cluster-level re-split, because training-side redundancy remains, but it is")
    print("a genuine sensitivity check and it needs no GPU.")
    print("\nIf BanglaSarc's diagonal barely moves, say so in Section 6.1 and delete the")
    print("'random-split' hedge in the Conclusion. If it drops materially, report both")
    print("numbers and recompute the retention denominators.")
else:
    print("No diagonal prediction files found under", PRED / "18_multiseed_cross_corpus")
    print("The cluster and leakage tables above are still valid and reportable on their own.")

print(f"\nWritten to {OUT}")

root: /Users/sefayet/Desktop/Github/Sarcasm_detection
threshold: 0.92

Ben-Sarc: 25623 rows  (train 20498, val 2562, test 2563)
  clusters 25494  multi-item 119  largest 4  test leaked 28/2563 (1.1%)

BanglaSarc: 4635 rows  (train 3708, val 463, test 464)
  clusters 4498  multi-item 123  largest 4  test leaked 25/464 (5.4%)

BanglaSarc3: 7910 rows  (train 6328, val 791, test 791)
  clusters 7894  multi-item 15  largest 3  test leaked 3/791 (0.4%)

CLUSTER STRUCTURE
            corpus  n_rows  n_clusters  n_multi_item_clusters  largest_cluster  rows_in_multi_item_clusters  pct_rows_in_clusters  near_dup_pairs
   ben_sarc_binary   25623       25494                    119                4                          248                  0.97             137
 banglasarc_binary    4635        4498                    123                4                          260                  5.61             146
banglasarc3_binary    7910        7894                     15                3              